# E2: train YOLO26s-pose 32 điểm mốc sân (SoccerNet) trên Google Colab, Hugging Face Hub làm kho

Một repo dataset private trên HF chứa `data/calibration/yolo-sn32/` (`train.zip`, `val.zip`, `data.yaml`) và `runs/pose/<RUN_NAME>/` (checkpoint, `results.csv`, biểu đồ). Colab chỉ cần token.

Chuẩn bị một lần (local): tạo `.env` từ `.env.example` (`HF_TOKEN` quyền write, `HF_REPO`) → `python scripts/hf_sync.py data` (nén + đẩy ~1.9 GB).
Colab: Secrets (🔑 bên trái) → thêm `HF_TOKEN` (cùng token), bật Notebook access. Runtime → Change runtime type → **GPU** (T4 đủ).

Checkpoint lên HF mỗi `SYNC_EVERY` epoch: session chết thì **Run all** lại, train tiếp từ `last.pt`. Train lại từ đầu: đổi `RUN_NAME`.
Thiết lập train cố định theo `research/calibration/experiments/e2-yolo-pose/protocol.md`.

## 0. Cấu hình

In [ ]:
from pathlib import Path

HF_REPO = 'LuxeFats/football-tracker'   # trùng HF_REPO trong .env
RUN_NAME = 'pitch-sn32-yolo26s'
RESUME = True     # có runs/pose/<RUN_NAME>/weights/last.pt trên HF thì train tiếp từ đó
SYNC_EVERY = 5    # đẩy checkpoint lên HF mỗi N epoch (session chết mất tối đa N epoch)

# Cố định theo protocol E2
TRAIN_ARGS = dict(
    imgsz=640, epochs=40, patience=10,
    batch=16,          # 8 nếu hết VRAM
    mosaic=0.0,        # điểm mốc ảo (ngoài khung hình) cần cả khung hình
    fliplr=0.0,        # lật góc đầu sân và góc cạnh sân cần hoán vị điểm khác nhau
    scale=0.3, translate=0.1,
)

WORK_DIR = Path('/content/football-tracker')
DATA_DIR = WORK_DIR / 'data/calibration/yolo-sn32'
DATA_YAML = DATA_DIR / 'data_colab.yaml'
RUN_DIR = WORK_DIR / 'runs/pose' / RUN_NAME

## 1. Cài Ultralytics, tải data (và run cũ để train tiếp) từ Hugging Face

In [ ]:
!pip install -q ultralytics==8.4.182   # trùng requirements.txt

import os
import zipfile

import yaml
from google.colab import userdata
from huggingface_hub import snapshot_download

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
patterns = ['data/calibration/yolo-sn32/*']
if RESUME:
    patterns.append(f'runs/pose/{RUN_NAME}/*')
snapshot_download(HF_REPO, repo_type='dataset', local_dir=WORK_DIR, allow_patterns=patterns)

for split in ('train', 'val'):
    done = DATA_DIR / f'.{split}_extracted'   # giải nén dở (session chết) thì giải nén lại
    if not done.exists():
        with zipfile.ZipFile(DATA_DIR / f'{split}.zip') as zf:
            zf.extractall(DATA_DIR)
        done.touch()
    print(split, sum(1 for _ in (DATA_DIR / 'images' / split).iterdir()), 'images')

cfg = yaml.safe_load((DATA_DIR / 'data.yaml').read_text())   # kpt_shape, flip_idx, names
cfg['path'] = str(DATA_DIR)                                   # data.yaml mang đường dẫn của máy local
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(DATA_YAML.read_text())

## 2. Train (checkpoint lên HF trong lúc train, chạy nền)

In [ ]:
import torch
from huggingface_hub import HfApi
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Bật GPU runtime trước (Runtime → Change runtime type → GPU)'
print('GPU:', torch.cuda.get_device_name(0))

api = HfApi()
uploads = []

def push(message, wait=False):
    """Đẩy weights + metrics + biểu đồ của run lên HF, chạy nền để không chặn train."""
    for f in uploads:
        if f.done() and f.exception():
            print('HF upload lỗi:', f.exception())
    uploads[:] = [f for f in uploads if not f.done()]
    uploads.append(api.upload_folder(
        repo_id=HF_REPO, repo_type='dataset', folder_path=RUN_DIR, path_in_repo=f'runs/pose/{RUN_NAME}',
        allow_patterns=['weights/last.pt', 'weights/best.pt', '*.csv', '*.yaml', '*.png', '*.jpg'],
        commit_message=f'{RUN_NAME}: {message}', run_as_future=True,
    ))
    if wait:
        for f in uploads:
            f.result()

def on_fit_epoch_end(trainer):   # Ultralytics gọi sau khi đã lưu last.pt / best.pt của epoch
    if (trainer.epoch + 1) % SYNC_EVERY == 0:
        push(f'epoch {trainer.epoch + 1}')

last = RUN_DIR / 'weights' / 'last.pt'
ckpt = torch.load(last, map_location='cpu', weights_only=False) if RESUME and last.exists() else None
# Train xong (kể cả dừng sớm vì patience), Ultralytics đặt epoch = -1 trong checkpoint.
if ckpt is not None and (ckpt['epoch'] == -1 or ckpt['epoch'] + 1 >= ckpt['train_args']['epochs']):
    print('Run đã train xong trên HF: bỏ qua train')
else:
    model = YOLO(str(last) if ckpt is not None else 'yolo26s-pose.pt')
    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
    if ckpt is not None:
        print(f"Train tiếp từ epoch {ckpt['epoch'] + 2}")
        model.train(resume=True)
    else:
        model.train(data=str(DATA_YAML), project=str(RUN_DIR.parent), name=RUN_NAME, exist_ok=True,
                    device=0, **TRAIN_ARGS)
    push('final', wait=True)
print(f'https://huggingface.co/datasets/{HF_REPO}/tree/main/runs/pose/{RUN_NAME}')

## 3. Kết quả trên SoccerNet valid (chọn checkpoint)

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(RUN_DIR / 'results.csv')
df.columns = df.columns.str.strip()
print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50(P)', 'metrics/mAP50-95(P)']].to_string(index=False))
for name in ('results.png', 'val_batch0_pred.jpg'):
    if (RUN_DIR / name).exists():
        display(Image(filename=str(RUN_DIR / name), width=900))

## 4. Đánh giá theo protocol E2 (máy local: TS-WorldCup, WC14, sample.mp4, SoccerNet test chỉ có ở local)

```
python scripts/hf_sync.py pull runs
PYTHONPATH=. .venv/Scripts/python.exe research/calibration/experiments/e2-yolo-pose/code/evaluate.py runs/pose/pitch-sn32-yolo26s/weights/best.pt
```